# AutoValue — Used Car Price Prediction

## Modelling and Evaluation

In this notebook, I develop regression models to predict used-car listing prices using vehicle characteristics.

The initial version of AutoValue focuses on the **mainstream used-car market**, with vehicles priced between **$1,000 and $100,000**. This modelling range was selected after exploratory analysis revealed that the extreme upper end of the dataset contains a mixture of erroneous listings and specialized luxury or collector vehicles with substantially different pricing behaviour.

I build the solution incrementally:

1. **Baseline Linear Regression** — using mileage as a single predictor.
2. **Multiple Linear Regression** — incorporating additional vehicle characteristics.
3. **Feature Engineering** — testing whether engineered features improve predictions.
4. **Polynomial Regression** — investigating nonlinear relationships in vehicle depreciation and pricing.
5. **Model Evaluation** — comparing models using MAE and RMSE on unseen test data.

The mileage-only model serves as the baseline. Each subsequent experiment is compared against this benchmark to determine whether increasing model complexity produces meaningful improvements in predictive performance.

In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder

In [2]:
df = pd.read_csv("../data/raw/cars.csv")

## 1. Modelling Data Preparation

I apply the data-preparation decisions identified during exploratory data analysis before training the models.

For the initial version of AutoValue, I focus on the mainstream used-car market and restrict the target price to between **$1,000 and $100,000**. The lower bound removes clearly implausible listing prices, while the upper bound defines the scope of the model and excludes specialized luxury and collector vehicles.

I also remove rows with missing mileage because mileage is an important predictor and only a very small proportion of observations are missing this value.

The same cleaned dataset will be used across all modelling experiments to ensure a fair comparison between models.

In [3]:
MIN_PRICE = 1_000
MAX_PRICE = 100_000

clean_df = df[
    (df["price"] >= MIN_PRICE) &
    (df["price"] <= MAX_PRICE)
].copy()

clean_df = clean_df.dropna(
    subset=["mileage"]
).copy()

In [4]:
REFERENCE_YEAR = 2024

clean_df["vehicle_age"] = (
    REFERENCE_YEAR - clean_df["year"]
)

In [5]:
clean_df["mileage_per_year"] = (
    clean_df["mileage"] /
    clean_df["vehicle_age"].clip(lower=1)
)

In [6]:
clean_df[
    [
        "year",
        "vehicle_age",
        "mileage",
        "mileage_per_year",
        "price"
    ]
].head()

,year,vehicle_age,mileage,mileage_per_year,price
0,2013,11,92945.0,8449.545455,13988.0
1,2013,11,47645.0,4331.363636,17995.0
2,2013,11,53422.0,4856.545455,17000.0
3,2013,11,117598.0,10690.727273,14958.0
4,2013,11,114865.0,10442.272727,14498.0


## 2. Baseline Feature and Target

I begin with a simple linear regression baseline using only vehicle mileage.

This provides a reference model that I can later compare against models using additional features.

For the baseline:

- `X` = mileage
- `y` = vehicle price

In [7]:
X = clean_df[["mileage"]]
y = clean_df["price"]
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (751667, 1)
y shape: (751667,)


## 3. Train-Test Split

To evaluate how well the model generalizes to unseen vehicle listings, I divide the dataset into training and test sets.

I use 80% of the observations for training and reserve 20% for testing. The model learns its parameters from the training data only, while the test set remains unseen until model evaluation.

I use a fixed `random_state` so that the same train-test split can be reproduced whenever the notebook is rerun.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [9]:
print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

print("y_train:", y_train.shape)
print("y_test: ", y_test.shape)

X_train: (601333, 1)
X_test:  (150334, 1)
y_train: (601333,)
y_test:  (150334,)


## 4. Baseline Linear Regression

I train my first baseline model using simple linear regression with `mileage` as the only predictor.

The model has the form:

\[
\hat{y} = wx + b
\]

where:

- \(x\) represents vehicle mileage
- \(w\) represents the slope or learned coefficient
- \(b\) represents the intercept
- \(\hat{y}\) represents the predicted vehicle price

Based on the exploratory analysis, I expect the learned coefficient for mileage to be negative because higher-mileage vehicles generally have lower prices.

In [10]:
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](1,)",[-0.21]
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](1,)",['mileage']
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,4.277e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,1
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(1)


In [11]:
print("Coefficient (w):", baseline_model.coef_[0])
print("Intercept (b):", baseline_model.intercept_)

Coefficient (w): -0.20970928269455003
Intercept (b): 42772.532445436795


## 5. Predictions on the Test Set

After training the baseline model, I use it to predict vehicle prices for the unseen test set.

The model receives only the mileage values from `X_test` and calculates predicted prices using the learned linear relationship:

\[
\hat{y} = wx + b
\]

I then compare these predictions with the actual prices in `y_test` to evaluate how well the model generalizes to unseen vehicles.

In [12]:
y_pred = baseline_model.predict(X_test)

In [13]:
results = pd.DataFrame({
    "mileage": X_test["mileage"],
    "actual_price": y_test,
    "predicted_price": y_pred
})

results.head(10)

,mileage,actual_price,predicted_price
106632,50956.0,15999.0,32086.586236
264006,40935.0,27900.0,34188.082958
229494,67032.0,49995.0,28715.299808
757067,4798.0,41527.0,41766.347307
511274,4689.0,36991.0,41789.205619
259250,42798.0,21024.0,33797.394565
354383,43442.0,20000.0,33662.341787
12878,61162.0,15299.0,29946.293297
706040,22008.0,14995.0,38157.250552
665755,4618.0,47999.0,41804.094978


In [14]:
mae = mean_absolute_error(y_test, y_pred)

print("Baseline MAE: $", round(mae, 2))

Baseline MAE: $ 10173.71


### RMSE — Root Mean Squared Error

I use RMSE alongside MAE to evaluate the baseline model. RMSE gives greater weight to large prediction errors because the errors are squared before being averaged.

The mileage-only baseline produced:

- **MAE:** $10,173.71
- **RMSE:** $13,728.05

This means the model's predictions differ from actual vehicle prices by approximately **$10,174 on average**. The higher RMSE indicates that some vehicles have considerably larger prediction errors.

This performance serves as the baseline against which I will compare more informative regression models.

In [15]:

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Baseline MAE: $", round(mae, 2))
print("Baseline RMSE: $", round(rmse, 2))

Baseline MAE: $ 10173.71
Baseline RMSE: $ 13728.05


In [16]:
largest_errors = results.copy()

largest_errors["absolute_error"] = abs(
    largest_errors["actual_price"] -
    largest_errors["predicted_price"]
)

largest_errors.sort_values(
    "absolute_error",
    ascending=False
).head(15)

,mileage,actual_price,predicted_price,absolute_error
494409,999999.0,22900.0,-166936.540540,189836.540540
463830,102501.0,99995.0,21277.121260,78717.878740
5438,99244.0,99999.0,21960.144394,78038.855606
635124,458906.0,22995.0,-53464.315639,76459.315639
527614,93948.0,97900.0,23070.764755,74829.235245
140782,79692.0,99000.0,26060.380289,72939.619711
545296,70025.0,99000.0,28087.639925,70912.360075
617110,68938.0,98981.0,28315.593915,70665.406085
616885,65704.0,98900.0,28993.793735,69906.206265
616437,91416.0,92500.0,23601.748659,68898.251341


In [17]:
print(
    "Maximum actual test price: $",
    y_test.max()
)

Maximum actual test price: $ 100000.0


In [18]:
df.nlargest(30, "price")[
    [
        "manufacturer",
        "model",
        "year",
        "mileage",
        "price"
    ]
]

,manufacturer,model,year,mileage,price
108142,Chevrolet,Cobalt LT,2009,85185.0,1.000000e+09
188113,Dodge,Durango Citadel,2018,113207.0,1.000000e+09
188260,Dodge,Durango Citadel,2018,113207.0,1.000000e+09
224571,Ford,Utility Police Interceptor Base,2016,NaN,8.888889e+06
84358,Cadillac,DeVille 77 HOURS ON ENGINES,1963,76.0,4.999999e+06
636956,RAM,ProMaster 3500 High Roof,2017,83000.0,3.490000e+06
613085,Porsche,Carrera GT,2005,780.0,2.250000e+06
609009,Porsche,918 Spyder Base (PDK),2015,2622.0,2.099995e+06
613080,Porsche,Carrera GT,2004,2361.0,1.899999e+06
49078,BMW,750 iL,1996,121043.0,1.750000e+06


In [19]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_pred)

print("Baseline MAE: $", round(mae, 2))
print("Baseline RMSE: $", round(rmse, 2))
print("Baseline R²:", round(r2, 4))

Baseline MAE: $ 10173.71
Baseline RMSE: $ 13728.05
Baseline R²: 0.3065


### Baseline Model Evaluation

The mileage-only linear regression model achieved:

- **MAE:** $10,173.71
- **RMSE:** $13,728.05
- **R²:** 0.3065

The model's predictions differ from actual vehicle prices by approximately $10,174 on average.

The RMSE is higher than the MAE, indicating that some observations have substantially larger prediction errors.

The R² score indicates that mileage alone explains approximately 30.7% of the variation in vehicle prices within the modelling dataset.

These results show that mileage contains useful predictive information, but it is insufficient on its own to accurately estimate vehicle prices. I therefore use this model as the baseline for evaluating more informative regression models.

## 7. Multiple Linear Regression

The baseline model uses mileage as its only predictor and explains approximately 30.7% of the variation in vehicle prices.

Vehicle age is another important factor in used-car valuation. I therefore extend the baseline model by using both `mileage` and `vehicle_age` as predictors.

The multiple linear regression model has the form:

\[
\hat{y} =
w_1(\text{mileage})
+
w_2(\text{vehicle age})
+
b
\]

I expect both coefficients to be negative because vehicle prices generally decrease as mileage and vehicle age increase.

The objective of this experiment is to determine whether adding vehicle age improves predictive performance compared with the mileage-only baseline.

In [21]:
X_multi = clean_df[
    ["mileage", "vehicle_age"]
]

y = clean_df["price"]

In [22]:
X_multi.head()

,mileage,vehicle_age
0,92945.0,11
1,47645.0,11
2,53422.0,11
3,117598.0,11
4,114865.0,11


In [23]:
#train/test split 
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi,
    y,
    test_size=0.20,
    random_state=42
)

In [24]:
# Train
multi_model = LinearRegression()

multi_model.fit(
    X_train_multi,
    y_train_multi
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[ -0.18,-478.43]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['mileage','vehicle_age']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,4.408e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


In [25]:
print("Mileage coefficient:", multi_model.coef_[0])
print("Vehicle age coefficient:", multi_model.coef_[1])
print("Intercept:", multi_model.intercept_)

Mileage coefficient: -0.1800268740842747
Vehicle age coefficient: -478.430098629839
Intercept: 44077.32212139963


In [26]:
#predict 
y_pred_multi = multi_model.predict(X_test_multi)

In [28]:
mae_multi = mean_absolute_error(
    y_test_multi,
    y_pred_multi
)

rmse_multi = np.sqrt(
    mean_squared_error(
        y_test_multi,
        y_pred_multi
    )
)

r2_multi = r2_score(
    y_test_multi,
    y_pred_multi
)

print("Multiple Regression MAE: $", round(mae_multi, 2))
print("Multiple Regression RMSE: $", round(rmse_multi, 2))
print("Multiple Regression R²:", round(r2_multi, 4))

Multiple Regression MAE: $ 9984.51
Multiple Regression RMSE: $ 13564.36
Multiple Regression R²: 0.323


### Multiple Linear Regression Evaluation

Adding `vehicle_age` improved the model compared with the mileage-only baseline.

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Mileage Only | $10,173.71 | $13,728.05 | 0.3065 |
| Mileage + Vehicle Age | $9,984.51 | $13,564.36 | 0.3230 |

The reduction in MAE and RMSE shows that vehicle age provides additional predictive information. However, the improvement is relatively small.

The R² score increased from 0.3065 to 0.3230, indicating that mileage and vehicle age together explain approximately 32.3% of the variation in vehicle prices.

This suggests that additional vehicle characteristics are required to explain a larger proportion of price differences.

## 8. Adding Manufacturer

The previous multiple linear regression model used mileage and vehicle age and achieved an R² of 0.3230.

However, vehicles with similar age and mileage can still have very different prices depending on their manufacturer.

I therefore add `manufacturer` as the next feature to investigate whether brand information improves price prediction.

In [31]:
X_brand = clean_df[
    ["mileage", "vehicle_age", "manufacturer"]
]

y = clean_df["price"]

In [32]:
X_brand.head()

,mileage,vehicle_age,manufacturer
0,92945.0,11,Acura
1,47645.0,11,Acura
2,53422.0,11,Acura
3,117598.0,11,Acura
4,114865.0,11,Acura


In [33]:
#split data 
X_train_brand, X_test_brand, y_train_brand, y_test_brand = train_test_split(
    X_brand,
    y,
    test_size=0.20,
    random_state=42
)

In [44]:
# create onehotencoder 
encoder = OneHotEncoder(
    handle_unknown="ignore",
     drop="first",
    sparse_output=False
    
)

In [45]:
# fit the encoder 
encoder.fit(
    X_train_brand[["manufacturer"]]
)

,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infre

In [46]:
encoder.categories_
len(encoder.categories_[0])

30

In [47]:
# tranform
manufacturer_train_encoded = encoder.transform(
    X_train_brand[["manufacturer"]]
)

In [48]:
manufacturer_train_encoded[0]

array([0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

In [49]:
X_train_brand.iloc[0]["manufacturer"]

'Ford'

In [50]:
encoder.get_feature_names_out(["manufacturer"])

array(['manufacturer_Audi', 'manufacturer_BMW', 'manufacturer_Buick',
       'manufacturer_Cadillac', 'manufacturer_Chevrolet',
       'manufacturer_Chrysler', 'manufacturer_Dodge', 'manufacturer_Ford',
       'manufacturer_GMC', 'manufacturer_Honda', 'manufacturer_Hyundai',
       'manufacturer_INFINITI', 'manufacturer_Jaguar',
       'manufacturer_Jeep', 'manufacturer_Kia', 'manufacturer_Land Rover',
       'manufacturer_Lexus', 'manufacturer_Lincoln', 'manufacturer_Mazda',
       'manufacturer_Mercedes-Benz', 'manufacturer_Mitsubishi',
       'manufacturer_Nissan', 'manufacturer_Porsche', 'manufacturer_RAM',
       'manufacturer_Subaru', 'manufacturer_Tesla', 'manufacturer_Toyota',
       'manufacturer_Volkswagen', 'manufacturer_Volvo'], dtype=object)

In [51]:
manufacturer_train_encoded = encoder.transform(
    X_train_brand[["manufacturer"]]
)

manufacturer_test_encoded = encoder.transform(
    X_test_brand[["manufacturer"]]
)

In [52]:
print(manufacturer_train_encoded.shape)
print(manufacturer_test_encoded.shape)

(601333, 29)
(150334, 29)


In [53]:
#Get the numerical training data
numeric_train = X_train_brand[
    ["mileage", "vehicle_age"]
].to_numpy()

numeric_test = X_test_brand[
    ["mileage", "vehicle_age"]
].to_numpy()

In [54]:
numeric_train.shape

(601333, 2)

In [55]:
#Combine numerical + encoded manufacturer
X_train_final = np.hstack([
    numeric_train,
    manufacturer_train_encoded
])

X_test_final = np.hstack([
    numeric_test,
    manufacturer_test_encoded
])

In [56]:
print("Training shape:", X_train_final.shape)
print("Test shape:", X_test_final.shape)

Training shape: (601333, 31)
Test shape: (150334, 31)


In [57]:
# Train Model 3
brand_model = LinearRegression()

brand_model.fit(
    X_train_final,
    y_train_brand
)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](31,)","[ -0.17, -566.78, 1864.43,..., -631.59,-8871.61, 1396.18]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,4.496e+04
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,31
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(30)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](31,)","[33744924.7 , 3296. , 244.94,..., 67.82, 53.14, 15.21]"


In [58]:
y_pred_brand = brand_model.predict(
    X_test_final
)

In [59]:
# evaluate 
mae_brand = mean_absolute_error(
    y_test_brand,
    y_pred_brand
)

rmse_brand = np.sqrt(
    mean_squared_error(
        y_test_brand,
        y_pred_brand
    )
)

r2_brand = r2_score(
    y_test_brand,
    y_pred_brand
)

print("Manufacturer Model MAE: $", round(mae_brand, 2))
print("Manufacturer Model RMSE: $", round(rmse_brand, 2))
print("Manufacturer Model R²:", round(r2_brand, 4))

Manufacturer Model MAE: $ 8547.31
Manufacturer Model RMSE: $ 11802.52
Manufacturer Model R²: 0.4874


### Manufacturer Model Evaluation

Adding manufacturer information produced a substantial improvement over the previous regression models.

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Mileage Only | $10,173.71 | $13,728.05 | 0.3065 |
| Mileage + Vehicle Age | $9,984.51 | $13,564.36 | 0.3230 |
| Mileage + Vehicle Age + Manufacturer | $8,547.31 | $11,802.52 | 0.4874 |

The manufacturer-enhanced model reduced both MAE and RMSE while increasing R² from 0.3230 to 0.4874.

This indicates that manufacturer contains substantial predictive information beyond mileage and vehicle age. Vehicles with similar age and mileage can have considerably different market values depending on their brand.

However, the model still explains less than half of the observed price variation, suggesting that additional vehicle characteristics may further improve predictive performance.